# SikaBook zipformer fine-tune (Kaggle T4)

Fine-tunes an English streaming zipformer (icefall `pruned_transducer_stateless7_streaming`)
on the Ashesi Financial Inclusion Speech Dataset (CC-BY-4.0) + FLEURS aka_gh,
then exports ONNX for sherpa-onnx.

Before running:
1. Run `data/datasets/ashesi/prepare_ashesi.py` locally; upload `prepared/` as a Kaggle Dataset named `sikabook-ashesi-prepared`.
2. Upload the base icefall checkpoint + tokens (see asr/zipformer/README.md) as `sikabook-zipformer-base`.
3. GPU: T4 (2x) or P100. Runtime budget: ~8-12 epochs fits comfortably in one 9h Kaggle session for ~150h of data with batch 16.
License gate: confirm docs/LICENSE-MATRIX.md before the first training run.

In [ ]:
# 1. Environment: icefall + deps (Kaggle images have torch; we pin what icefall needs)
!pip -q install lhotse==1.19.2 k2==1.24.4-dev20240223+cuda11.8.torch2.1.2 -f https://k2-fsa.github.io/k2/cpu.html || true
!git clone --depth 1 https://github.com/k2-fsa/icefall.git /kaggle/working/icefall
%cd /kaggle/working/icefall
!pip -q install -r requirements.txt || true

In [ ]:
# 2. Data: manifest.jsonl (from prepare_ashesi.py) -> lhotse cuts
import json, glob
from pathlib import Path
from lhotse import CutSet, Recording, MonoCut
from lhotse.supervision import SupervisionSegment

manifests = glob.glob('/kaggle/input/sikabook-ashesi-prepared/*/manifest.jsonl')
assert manifests, 'upload prepared manifests as a Kaggle dataset first'

cuts = []
for m in manifests:
    lang_dir = Path(m).parent
    with open(m) as f:
        for line in f:
            r = json.loads(line)
            rec = Recording.from_file(r['audio'])
            sup = SupervisionSegment(id=rec.id, recording_id=rec.id,
                                     start=0.0, duration=rec.duration,
                                     text=r['transcript'], language=r['language'],
                                     speaker=r['speaker'])
            cuts.append(MonoCut(id=rec.id, recording=rec, supervisions=[sup],
                                start=0.0, duration=rec.duration))
cutset = CutSet.from_cuts(cuts)
cutset = cutset.filter(lambda c: 0.4 <= c.duration <= 20.0)
cutset.to_file('/kaggle/working/ashesi_cuts.jsonl.gz')
print(f'{len(cutset)} cuts, {cutset.duration/3600:.1f} hours')

In [ ]:
# 3. Tokens: extend base tokens.txt with Akan letters if not present
base_tokens = '/kaggle/input/sikabook-zipformer-base/tokens.txt'
tokens = open(base_tokens, encoding='utf-8').read().splitlines()
vocab = {t.split()[0] for t in tokens if t.strip()}
max_id = max(int(t.split()[1]) for t in tokens if t.strip())
extra = [c for c in ['\u025b', '\u0254'] if c not in vocab]  # ɛ ɔ
with open('/kaggle/working/tokens.txt', 'w', encoding='utf-8') as f:
    for t in tokens:
        f.write(t + '\n')
    for i, c in enumerate(extra):
        f.write(f'{c} {max_id + 1 + i}\n')
print('added', extra, '- OR use --normalize-ascii prep to avoid token surgery')
print('NOTE: adding output units requires embedding warm-up; first run should'
      ' use ASCII-normalized transcripts instead (prepare_ashesi.py --normalize-ascii)')

In [ ]:
# 4. Fine-tune (streaming transducer, single GPU)
%cd /kaggle/working/icefall/egs/librispeech/ASR
!python pruned_transducer_stateless7_streaming/train.py \
  --world-size 1 \
  --num-epochs 10 \
  --start-epoch 0 \
  --exp-dir /kaggle/working/exp_finetune \
  --base-lr 0.0001 \
  --finetune True \
  --finetune-checkpoint /kaggle/input/sikabook-zipformer-base/checkpoint.pt \
  --manifest-dir /kaggle/working \
  --training-subset ashesi_cuts 2>&1 | tail -20
# (recipe arg names shift between icefall versions - check the recipe's
#  train.py argparse and the finetune notes in asr/zipformer/README.md)

In [ ]:
# 5. Decode dev split, compute WER with the repo's own harness later
!python pruned_transducer_stateless7_streaming/decode.py \
  --exp-dir /kaggle/working/exp_finetune \
  --epoch 10 --avg 3 \
  --manifest-dir /kaggle/working \
  --decoding-method modified_beam_search 2>&1 | tail -10

In [ ]:
# 6. Export ONNX -> download -> int8 quantize locally -> sherpa-onnx
!python pruned_transducer_stateless7_streaming/export-onnx.py \
  --exp-dir /kaggle/working/exp_finetune \
  --epoch 10 --avg 3 --streaming True 2>&1 | tail -5
!ls -la /kaggle/working/exp_finetune/*.onnx
# Download encoder/decoder/joiner onnx + tokens.txt, then run
# export/package_android.sh locally to build the sherpa-onnx bundle.